## Inference under FHE for the MNIST Dataset using helayers

In this demo, we'll deal with a classification problem for the MNIST dataset [1], trying to correctly classify a batch of samples using a neural network model that will be created and trained using the Keras library (with architecture similar to reference [2]).
First, we'll build a plain neural network for the MNIST model. Then, we'll encrypt the trained network and run inference over it using FHE.

In [1]:
import os

##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)
from tensorflow.keras import backend as K

from tensorflow.keras import utils, losses
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
import h5py

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)

# batch_size = 16
# batch_size = 32
# batch_size = 64
# batch_size = 128
# batch_size = 256
# batch_size = 512
batch_size = 1024
# batch_size = 2048
# batch_size = 4096
# batch_size=8192
print("Misc. initializations")

2025-05-30 06:57:00.032750: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Misc. initializations


In [2]:
import numpy as np
import gzip

def load_images(path):
    """加载图像文件（idx3-ubyte格式）"""
    with gzip.open(path, 'rb') as f:
        # 图像文件前16字节为幻数（4字节）、图像数量（4字节）、行数（4字节）、列数（4字节）
        data = np.frombuffer(f.read(), np.uint8, offset=16)
    return data.reshape(-1, 28, 28)  # 重塑为（样本数, 28, 28）

def load_labels(path):
    """加载标签文件（idx1-ubyte格式）"""
    with gzip.open(path, 'rb') as f:
        # 标签文件前8字节为幻数（4字节）、标签数量（4字节）
        data = np.frombuffer(f.read(), np.uint8, offset=8)
    return data  # 直接返回一维数组，无需reshape

# 加载训练和测试数据
x_train = load_images('/workspace/examples/vision_transformer/data/KMNIST/train-images-idx3-ubyte.gz')
y_train = load_labels('/workspace/examples/vision_transformer/data/KMNIST/train-labels-idx1-ubyte.gz')
x_test = load_images('/workspace/examples/vision_transformer/data/KMNIST/t10k-images-idx3-ubyte.gz')
y_test = load_labels('/workspace/examples/vision_transformer/data/KMNIST/t10k-labels-idx1-ubyte.gz')

### Load and Preprocess the MNIST Dataset. 

In [3]:


# 以下预处理步骤与MNIST完全兼容，无需修改
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# 裁剪部分（根据需求选择性取消注释）
# x_train = x_train[:, 2:26, 2:26]
# x_test = x_test[:, 2:26, 2:26]

# 添加通道维度（Fashion-MNIST为灰度图，需扩展为[28,28,1]）
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# 归一化至[0,1]范围
x_train /= 255
x_test /= 255

print('Fashion-MNIST data ready')

Fashion-MNIST data ready


In [4]:
# Create validation data
# valsize=16
# valsize=500
valsize=5000
x_train = x_train[:-valsize]
x_val = x_train[-valsize:]
y_train = y_train[:-valsize]
y_val = y_train[-valsize:]
print('Validation and test data ready')

# Convert class vector to binary class matrices
# num_classes = 10
# y_train = utils.to_categorical(y_train, num_classes)
# y_test = utils.to_categorical(y_test, num_classes)
# y_val = utils.to_categorical(y_val, num_classes)

input_shape = x_train[0].shape
print(f'input shape: {input_shape}')

Validation and test data ready
input shape: (28, 28, 1)


### Save Dataset

In [5]:
def save_data_set(x, y, data_type, s=''):
    fname=os.path.join(PATH, f'x_{data_type}{s}.h5')
    print("Saving x_{} of shape {} in {}".format(data_type, x.shape,fname))
    xf = h5py.File(fname, 'w')
    xf.create_dataset('x_{}'.format(data_type), data=x)
    xf.close()

    yf = h5py.File(os.path.join(PATH, f'y_{data_type}{s}.h5'), 'w')
    yf.create_dataset(f'y_{data_type}', data=y)
    yf.close()

save_data_set(x_test, y_test, data_type='test9')
save_data_set(x_train, y_train, data_type='train')
save_data_set(x_val, y_val, data_type='val')

Saving x_test9 of shape (10000, 28, 28, 1) in data/net_mnist/x_test9.h5
Saving x_train of shape (55000, 28, 28, 1) in data/net_mnist/x_train.h5
Saving x_val of shape (5000, 28, 28, 1) in data/net_mnist/x_val.h5


### Build a Plain Neural Network for the MNIST Model

In [6]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D,AveragePooling2D,Softmax
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2

# 创建模型
model = Sequential()
# 第一层卷积层（无激活函数）
model.add(Conv2D(20, (3, 3), input_shape=(28, 28, 1),kernel_regularizer=l2(0.0005)))  # 移除activation参数


# AveP-BN操作
model.add(AvgPool2D(pool_size=2))
model.add(BatchNormalization())

# 第二层卷积层（无激活函数）
model.add(Conv2D(50, (3, 3),kernel_regularizer=l2(0.0005)))  # 移除activation参数
model.add(ReLU())  # 显式添加激活层

# BN操作
model.add(BatchNormalization())
model.add(Flatten())
# 全连接层，500个神经元
model.add(Dense(500))

# 输出层，10个神经元（假设是10分类任务）
model.add(Dense(10,kernel_regularizer=l2(0.0005)))


model.summary()

2025-05-30 06:57:04.912252: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-05-30 06:57:04.951729: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-05-30 06:57:04.954629: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 20)        200       
                                                                 
 average_pooling2d (AverageP  (None, 13, 13, 20)       0         
 ooling2D)                                                       
                                                                 
 batch_normalization (BatchN  (None, 13, 13, 20)       80        
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 50)        9050      
                                                                 
 re_lu (ReLU)                (None, 11, 11, 50)        0         
                                                                 
 batch_normalization_1 (Batc  (None, 11, 11, 50)       2

2025-05-30 06:57:04.958107: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-30 06:57:04.958788: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-05-30 06:57:04.961630: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-05-30 06:57:04.964223: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so ret

### Train the Neural Network

In [7]:
#一阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow_addons as tfa

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=10,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
# optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
def lr_scheduler(epoch):
    if epoch < 100:
        return 0.001
    elif epoch < 200:
        return 0.0005
    else:
        return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

model.fit(
    x_train, y_train, batch_size=batch_size,
    
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/root/miniforge3/lib/python3.10/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.9.1 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make su

Epoch 1/4000


2025-05-30 06:57:08.146413: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8500


54/54 - 4s - loss: 0.9762 - accuracy: 0.8437 - val_loss: 1.9665 - val_accuracy: 0.4000 - 4s/epoch - 73ms/step
Epoch 2/4000
54/54 - 1s - loss: 0.2391 - accuracy: 0.9467 - val_loss: 2.5545 - val_accuracy: 0.1908 - 909ms/epoch - 17ms/step
Epoch 3/4000
54/54 - 1s - loss: 0.1556 - accuracy: 0.9625 - val_loss: 3.6788 - val_accuracy: 0.2034 - 907ms/epoch - 17ms/step
Epoch 4/4000
54/54 - 1s - loss: 0.1192 - accuracy: 0.9712 - val_loss: 3.2363 - val_accuracy: 0.2974 - 906ms/epoch - 17ms/step
Epoch 5/4000
54/54 - 1s - loss: 0.0907 - accuracy: 0.9782 - val_loss: 2.7697 - val_accuracy: 0.3348 - 917ms/epoch - 17ms/step
Epoch 6/4000
54/54 - 1s - loss: 0.0673 - accuracy: 0.9854 - val_loss: 2.3942 - val_accuracy: 0.3800 - 909ms/epoch - 17ms/step
Epoch 7/4000
54/54 - 1s - loss: 0.0569 - accuracy: 0.9877 - val_loss: 1.4188 - val_accuracy: 0.5538 - 914ms/epoch - 17ms/step
Epoch 8/4000
54/54 - 1s - loss: 0.0496 - accuracy: 0.9910 - val_loss: 0.6157 - val_accuracy: 0.7858 - 917ms/epoch - 17ms/step
Epoch 9/

### Rebuild Model

In [8]:
from tensorflow.keras.layers import Dense, Flatten, Conv2D, Activation, BatchNormalization, ReLU, AvgPool2D
from tensorflow.keras.layers import GlobalAvgPool2D
from data_gen.activations import SquareActivation
from tensorflow.keras.regularizers import l2
new_model = Sequential()
for layer in model.layers:
    
    if isinstance(layer, ReLU):
        new_model.add(SquareActivation())
    else:
        new_layer = layer.__class__.from_config(layer.get_config())
        new_model.add(new_layer)
        if layer.weights:
            new_layer.set_weights(layer.get_weights())  # 自动复制权重和偏置

new_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 20)        200       
                                                                 
 average_pooling2d (AverageP  (None, 13, 13, 20)       0         
 ooling2D)                                                       
                                                                 
 batch_normalization (BatchN  (None, 13, 13, 20)       80        
 ormalization)                                                   
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 50)        9050      
                                                                 
 square_activation (SquareAc  (None, 11, 11, 50)       0         
 tivation)                                                       
                                                      

In [9]:
#二阶段
epochs = 4000
# epochs =1
# epochs = 3000
# epochs = 500
from tensorflow.keras.callbacks import EarlyStopping

# 添加早停回调
early_stopping = EarlyStopping(
    monitor='val_loss',  # 监控验证准确率
    
    patience=10,         # 如果20个epoch没有改善则停止
    restore_best_weights=True  # 恢复最佳模型权重
)
def sum_squared_error(y_true, y_pred):
    return K.sum(K.square(y_pred - y_true), axis=-1)
base_learning_rate=0.01
class CustomLearningRateSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_learning_rate):
        self.base_learning_rate = base_learning_rate

    def __call__(self, epoch):
        return self.base_learning_rate *(1+(1e-4) * (epoch)) ** (-0.75)
    # 创建自定义的学习率调度器

# learning_rate_schedule = CustomLearningRateSchedule(base_learning_rate)
# optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate_schedule, momentum=0.9)
# optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 阶梯式学习率衰减
def lr_scheduler(epoch):
    if epoch < 100:
        return 0.001
    elif epoch < 200:
        return 0.0005
    else:
        return 0.0001
callbacks = [
    # tf.keras.callbacks.LearningRateScheduler(lr_scheduler),
    early_stopping
]
loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),

new_model.compile(loss=loss,
                  optimizer=optimizer,
                  metrics=['accuracy'])

new_model.fit(
    x_train, y_train, batch_size=batch_size,
   
              epochs=epochs,
              verbose=2,
              validation_data=(x_val, y_val),
              shuffle=True
              ,
              callbacks=callbacks
              )

score = new_model.evaluate(x_test, y_test, verbose=2)

print(f'Test loss: { score[0]:.3f}')
print(f'Test accuracy: {score[1] * 100:.3f}%')

Epoch 1/4000
54/54 - 2s - loss: 0.4428 - accuracy: 0.8985 - val_loss: 6.4879 - val_accuracy: 0.4974 - 2s/epoch - 34ms/step
Epoch 2/4000
54/54 - 1s - loss: 0.1197 - accuracy: 0.9632 - val_loss: 0.6890 - val_accuracy: 0.8570 - 1s/epoch - 20ms/step
Epoch 3/4000
54/54 - 1s - loss: 0.0933 - accuracy: 0.9705 - val_loss: 0.1356 - val_accuracy: 0.9610 - 1s/epoch - 20ms/step
Epoch 4/4000
54/54 - 1s - loss: 0.0756 - accuracy: 0.9772 - val_loss: 0.0588 - val_accuracy: 0.9816 - 1s/epoch - 20ms/step
Epoch 5/4000
54/54 - 1s - loss: 0.0635 - accuracy: 0.9806 - val_loss: 0.0470 - val_accuracy: 0.9852 - 1s/epoch - 20ms/step
Epoch 6/4000
54/54 - 1s - loss: 0.0490 - accuracy: 0.9853 - val_loss: 0.0368 - val_accuracy: 0.9894 - 1s/epoch - 20ms/step
Epoch 7/4000
54/54 - 1s - loss: 0.0399 - accuracy: 0.9883 - val_loss: 0.0342 - val_accuracy: 0.9904 - 1s/epoch - 20ms/step
Epoch 8/4000
54/54 - 1s - loss: 0.0346 - accuracy: 0.9901 - val_loss: 0.0287 - val_accuracy: 0.9926 - 1s/epoch - 20ms/step
Epoch 9/4000
54/

### Serialize Model and Weights

In [10]:
model_json = new_model.to_json()
with open(os.path.join(PATH, 'model113.json'), "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
new_model.save_weights(os.path.join(PATH, 'model113.h5'))
print("Saved model to disk")

Saved model to disk


We are all done training the plain network. Next we will encrypt the network and run inference over it using FHE. Let's start with some initializations.

In [ ]:
import pyhelayers
import utils

utils.verify_memory()

print('Misc. initializations')

In [ ]:
import pyhelayers
import utils
import h5py
import os
import numpy as np
##### For reproducibility
seed_value= 1
os.environ['PYTHONHASHSEED']=str(seed_value)

# import activations
import sys
sys.path.append(os.path.join('.', 'data_gen'))
# from activations import SquareActivation

PATH = os.path.join('data', 'net_mnist')
if not os.path.exists(PATH):
    os.makedirs(PATH)



utils.verify_memory()

print('Misc. initializations')

batch_size=500

The following is a general outline of the next steps.

We encode and encrypt a neural network model, using the files that were created and saved before. An automated optimizer, that occurs during the call to encode_encrypt, will examine our network and will determine various configuration details that will allow running inference under encryption efficiently.

Next, we will demonstrate how we can encrypt data, run inference on our encrypted network, and compare the results against the expected labels.
Now let's dive in . . .

In [ ]:
he_run_req = pyhelayers.HeRunRequirements()
he_run_req.set_he_context_options([pyhelayers.DefaultContext()])
he_run_req.optimize_for_batch_size(16)

nn = pyhelayers.NeuralNet()
nn.encode_encrypt([os.path.join(PATH, "model.json"), os.path.join(PATH, "model.h5")], he_run_req)

The encode_encrypt method also initialized an HeContext object containing the keys. We obtain it now from the model so we can encrypt the input data.

In [ ]:
context = nn.get_created_he_context()

We will now load real samples of the MNIST dataset to classify. We will load the samples and the corresponding true labels from HDF5 files. We will also extract the first batch of samples and labels.

In [ ]:
with h5py.File(os.path.join(PATH, "x_test.h5")) as f:
    x_test = np.array(f["x_test"])
with h5py.File(os.path.join(PATH, "y_test.h5")) as f:
    y_test = np.array(f["y_test"])
    
# plain_samples, labels = utils.extract_batch(x_test, y_test, batch_size, 0)

plain_samples, labels = utils.extract_batch(x_test, y_test, 100, 0)
print('Batch of size',batch_size,'loaded')

To populate the input, we need to encode and then encrypt the values of the plain input under HE.

In [ ]:
model_io_encoder = pyhelayers.ModelIoEncoder(nn)
samples = pyhelayers.EncryptedData(context)
model_io_encoder.encode_encrypt(samples, [plain_samples])
print('Test data encrypted')

We now go ahead with the inference itself. We run the encrypted input through the encrypted NN to obtain encrypted results. This computation does not use the secret key and acts on completely encrypted values. Running the inference is done using the "predict" method of the NN, that receives the destination 3D structure to put the result of the computation in, and the input for the inference.

In [ ]:
utils.start_timer()

predictions = pyhelayers.EncryptedData(context)
nn.predict(predictions, samples)

duration=utils.end_timer('predict')
utils.report_duration('predict per sample',duration/batch_size)

In order to assess the results of the computation, we first need to decrypt them. This is done by an IO processor that has the secret key and is capable of decrypting the ciphertext and decoding it into plaintext version of the HE computation result.

In [ ]:
plain_predictions = model_io_encoder.decrypt_decode_output(predictions)
print('predictions',plain_predictions)

Now we compare the results against the expected labels and compute the confusion matrix and the accuracy.

In [ ]:
utils.assess_results(labels, plain_predictions)

<br>

References:

<sub><sup> 1.	LeCun, Yann and Cortes, Corinna. "MNIST handwritten digit database." (2010): </sup></sub>

<sub><sup> 2.	Gilad-Bachrach, R., Dowlin, N., Laine, K., Lauter, K., Naehrig, M. &amp; Wernsing, J.. (2016). CryptoNets: Applying Neural Networks to Encrypted Data with High Throughput and Accuracy. Proceedings of The 33rd International Conference on Machine Learning, in Proceedings of Machine Learning Research 48:201-210 Available from https://proceedings.mlr.press/v48/gilad-bachrach16.html.
</sup></sub>
